# Solving the Gouy–Chapman–Stern model
We seek to solve the Gouy–Chapman–Stern model of the symmetric electrolyte 1:1 model:
$$
\boxed{
\begin{aligned}
  \text{Helmholtz} &\begin{cases} \frac{\text{d}^{2}\phi}{\text{d}x^{2}} &= 0,           &\qquad
  &0 < x < x_{\text{HP}}  \\
  \phi\left(x\right) &= \phi_{\text{M}} - \phi_{\text{pzc}}, &  &x = 0 \\
  \phi\left(x\right) &= \phi_{\text{HP}},&   &x = x_{\text{HP}} \\
  \end{cases}
  & & \\
    \text{Gouy--Chapman}&\begin{cases}\varepsilon_{0}\varepsilon_{r}\frac{\text{d}^{2}\phi}{\text{d}x^{2} } &= -\frac{e_{0} N_{\mathrm{A}} c_{\mathrm{b}}}{2}\sinh\left(\frac{e_{0}\phi}{k_{\mathrm{B}}T}\right),  &x_{\text{HP}} < x < +\infty  \\
  \phi\left(x\right) &= \phi_{\text{M}} - \phi_{\text{pzc}} + \delta_{\text{HP}}\frac{\varepsilon_{\text{Solution}}}{\varepsilon_{\text{HP}}}\frac{\text{d} \phi}{\text{d} x }\big|_{x = x_{\text{HP}} ^{+}},  &x = x_{\text{HP}} \\ 
  \phi\left(x\right) &= 0,     &x = +\infty
    \end{cases}
\end{aligned}
}
$$


# Parametric setup  ###

### The following lines of codes set up the paramters and fundamental constants of the Gouy–Chapman–Stern model



In [25]:
from dataclasses import dataclass
import math

# Fundamental constants
e = 1.602176634e-19      # C
kB = 1.380649e-23        # J/K
eps0 = 8.8541878128e-12  # F/m
NA = 6.02214076e23       # 1/mol

@dataclass
class GCS_SI_Params:
    delta_H: float          # Distance to the Helmholtz Plane (m)
    eps_H: float            # Relative permittivity in the Stern layer
    eps_S: float = 78.5     # Relative permittivity of the bulk (relative)
    c_bulk: float = 1.0   # Debye length (m)
    T: float = 298.15       # Temperature (K)
    tol: float = 1e-12      # Tolerance before convergence 
    maxiter: int = 200      # Maximum iterations for solver in case anything goes wrong


def compute_lambda_D(p: GCS_SI_Params) -> float:
    return math.sqrt(
        p.eps_S * eps0 * kB * p.T /
        (2.0 * e**2 * NA * p.c_bulk)
    )

def validate_params_si(p: GCS_SI_Params) -> None:
    if p.T <= 0:
        raise ValueError("T must be > 0")
    if p.delta_H <= 0:
        raise ValueError("delta_H must be > 0")
    if p.eps_H <= 0 or p.eps_S <= 0:
        raise ValueError("eps_H and eps_S must be > 0")
    if p.c_bulk <= 0:
        raise ValueError("c_bulk must be > 0")

# Helper functions

### For stability, we define the following functions

$$
\begin{align*}
V_{\mathrm{T}} = \frac{ k_{\mathrm{B}}\phi }{e_{0}} \iff \phi = \frac{e_{0} V_{\mathrm{T}}}{k_{\mathrm{B}}}
\end{align*}
$$
with $V_{\mathrm{T}}$ being the thermal voltage.

In [26]:
def thermal_voltage(p: GCS_SI_Params) -> float:
    return kB * p.T / e


def V_to_U(p: GCS_SI_Params, V: float) -> float:
    return V / thermal_voltage(p)


def U_to_V(p: GCS_SI_Params, U: float) -> float:
    return U * thermal_voltage(p)

# Surface Free Charge Density and Double Layer Capacitance

## Surface Free Charge Density

$$
    \begin{aligned}
        \sigma_{\mathrm{free}}^{\left(\mathrm{H}\right)}\left( \phi_{\mathrm{M}} - \phi_{\mathrm{HP}} \right)     &= \frac{\varepsilon_{0}\varepsilon_{r}\left(\phi_{\mathrm{M}} - \phi_{\mathrm{HP}}\right) }{\delta_{\text{HP}}}  \\
        \sigma_{\mathrm{free}}^{\left(\mathrm{GC}\right)}\left(\phi_{\mathrm{HP}} - \phi_{\mathrm{pzc}}\right)    &= \sqrt{ 8 \varepsilon_{0}\varepsilon_{r} N_{\mathrm{A}} k_{\mathrm{B}} T c_{\mathrm{b}} } \text{sinh}\left(\frac{\beta e_{0} \left(\phi_{\mathrm{HP}} - \phi_{\mathrm{pzc}}\right) }{2 k_{\mathrm{B}} T } \right) \\
        \sigma_{\mathrm{free}}^{\left(\mathrm{GCS}\right)}\left(\phi_{\mathrm{M}} - \phi_{\mathrm{pzc}}\right)   &= \underbrace{\sigma_{\mathrm{free}}^{\left(\mathrm{H}\right)}\left( \phi_{\mathrm{M}} - \phi_{\mathrm{HP}} \right) = \sigma_{\mathrm{free}}^{\left(\mathrm{GC}\right)}\left(\phi_{\mathrm{HP}} - \phi_{\mathrm{pzc}}\right) }_{ \mathrm{We}\, \mathrm{solve} \, \mathrm{for} \,  \phi_{\mathrm{HP}} \, \mathrm{using} \, \mathrm{this} \, \mathrm{relation}}
    \end{aligned}
$$

## Double Layer Capacitance

This set of functions calculate the double layer capacitance, $C_{\mathrm{dl}}$, of the Helmholtz (H), Gouy–Chapman (GC), and Gouy–Chapman--Stern (GCS) capacitance profiles
$$
    \begin{aligned}
        C^{\left(\mathrm{H}\right)}\     &= \frac{\varepsilon_{0}\varepsilon_{r}}{\delta_{\text{H}}}  \\
        C^{\left(\mathrm{GC}\right)}\left(\phi_{\mathrm{HP}} - \phi_{\mathrm{pzc}}\right)    &= \varepsilon_{0}\varepsilon_{r} \lambda_{\text{D}}^{-1} \text{cosh}\left(\frac{\beta e_{0} \left(\phi_{\mathrm{M}} - \phi_{\mathrm{HP}}\right) }{2 k_{\mathrm{B}} T } \right) \\
        C^{\left(\mathrm{GCS}\right)}\left(\phi_{\mathrm{M}} - \phi_{\mathrm{pzc}}\right)   &= \left(\frac{1}{C^{\left(\mathrm{H}\right)} } + \frac{1}{C^{\left(\mathrm{GC}\right)}\left(\phi_{\mathrm{HP}} - \phi_{\mathrm{pzc}}\right)}\right)^{-1}
    \end{aligned}
$$

In [27]:
import math
import numpy as np

# Helmholtz
def C_H(p: GCS_SI_Params) -> float:
    return p.eps_H * eps0 / p.delta_H

# Gouy–Chapman
def C_GC(p: GCS_SI_Params, V_HP):
    """Diffuse-layer differential capacitance in F/m^2.

    Accepts either a scalar Helmholtz-plane potential or an array.
    """
    kappa = 1.0 / compute_lambda_D(p)
    V_HP_arr = np.asarray(V_HP, dtype=float)
    cap = p.eps_S * eps0 * kappa * np.cosh(e * V_HP_arr / (2.0 * kB * p.T))
    if np.ndim(V_HP) == 0:
        return float(cap)
    return cap

# Gouy–Chapman–Stern
def C_total(C_H: float, C_GC_HP):
    return 1.0 / (1.0 / C_H + 1.0 / C_GC_HP)


# Individual solutions to the models

\begin{aligned}
    \phi^{\left(\mathrm{GCS}\right)}\left(x\right) &= \begin{cases} \phi^{\left(\mathrm{H}\right)}\left(x\right) &= \frac{ \phi_{\mathrm{HP}} x}{\delta_{\mathrm{HP}}} + \left(\phi_{\mathrm{M}} - \phi_{\mathrm{pzc}}\right)\left(1 - \frac{x}{\delta_{\mathrm{HP}}}\right), \qquad &x \leq \delta_{HP} \\
                        \phi^{\left(\mathrm{GC}\right)}\left(x\right) &= 4 k_{\text{B}}T e_{0}^{-1}\,\text{arctanh}\left(\tanh\left(\frac{e_{0}\left(\phi_{\text{HP}} - \phi_{\text{pzc}}\right)  }{4k_{\text{B}}T }\right)\exp\left(-\lambda_{\text{D}}^{-1}\mathbf{x} \right) \right), \qquad &x > \delta_{\mathrm{HP}}
    \end{cases} \\ 
    c_{-}^{\left(\mathrm{GCS}\right)}\left(x\right) &= c_{\mathrm{b}}\, \exp\left(\frac{e_{0} \phi\left(x\right) }{k_{\mathrm{B}} T }\right), \qquad &x > \delta_{\mathrm{HP}} \\
    c_{+}^{\left(\mathrm{GCS}\right)}\left(x\right) &= c_{\mathrm{b}}\, \exp\left(-\frac{e_{0} \phi\left(x\right) }{k_{\mathrm{B}} T }\right), \qquad &x > \delta_{\mathrm{HP}}
\end{aligned}

with $$\lambda_{\text{D}}^{-1} = \left(\frac{2e_{0}^{2} n_{0} }{\varepsilon_{0}\varepsilon_{r} k_{\text{B}}T  }\right)^{\frac{1}{2}}$$ being the inverse Debye length.

In [28]:
import math

# Helmholtz
def U_H_scalar(x: float, delta_HP: float, U_M: float, U_HP: float) -> float:
    return U_HP * (x / delta_HP) + U_M * (1.0 - x / delta_HP)


# Gouy–Chapman
def U_GC_scalar(X: float, U0: float) -> float:
    return 4.0 * math.atanh(math.exp(-X) * math.tanh(U0 / 4.0))


# Gouy–Chapman–Stern
def U_GCS_scalar(x: float, delta_HP: float, lambda_D: float,
                 U_M: float, U_HP: float) -> float:
    if x <= delta_HP:
        return U_H_scalar(x, delta_HP, U_M, U_HP)
    else:
        X = (x - delta_HP) / lambda_D
        return U_GC_scalar(X, U_HP)


# Solution scheme

The solution scheme is based in the following. The electric potential at the Helmholtz Plane (HP) is always between the applied potential, $\phi_{\mathrm{M}}$, and $0$:
$$
\begin{aligned}
    \phi_{\mathrm{pzc}} \leq \phi_{\mathrm{HP}} \leq \phi_{\mathrm{M}}
\end{aligned}
$$
and the residual function
$$
\begin{aligned}
    F\left(\phi_{\mathrm{HP}}^{\left(\mathrm{guess}\right)}\right) &= \sigma_{\mathrm{free}}^{\left(\mathrm{H}\right)}\left( \phi_{\mathrm{M}} - \phi_{\mathrm{HP}}^{\left(\mathrm{guess}\right)} \right) - \sigma_{\mathrm{free}}^{\left(\mathrm{GC}\right)}\left(\phi_{\mathrm{HP}}^{\left(\mathrm{guess}\right)} - \phi_{\mathrm{pzc}}\right) \implies \\
    F\left(\phi_{\mathrm{HP}}^{\left(\mathrm{guess}\right)}\right) &= \phi_{\mathrm{M}} - \phi_{\mathrm{HP}}^{\left(\mathrm{HP}\right)} - 2 \frac{\varepsilon_{\text{r}}}{\varepsilon_{\mathrm{HP}}}\frac{\delta_{\mathrm{HP}} }{ \lambda_{\mathrm{D}}}\frac{k_{\mathrm{B}}T}{e_{0}} \sinh\left(\frac{e_{0}  \left(\phi_{\mathrm{HP}} - \phi_{\mathrm{pzc}}\right) }{2 k_{\mathrm{B}}T}\right)
\end{aligned}
$$

If $F\left(\phi_{\mathrm{HP}}^{\left(\mathrm{guess}\right)}\right) = 0$, then we have found $\phi_{\mathrm{HP}}^{\left(\mathrm{guess}\right)}$ has been found, if $F > 0$, then our $\phi^{\left(\mathrm{guess}\right)}_{\mathrm{HP}}$ is a new upper bound and vice versa. To this end, we simply iterate using 
$$
\begin{aligned}
    \phi^{\left(i + 1\right)} &= \frac{\phi^{\left(i\right)}_{LL} + \phi^{\left(i\right)}_{\mathrm{UP}} }{2} \\
    F\left(\phi^{\left(i+1\right)}\right)> \epsilon &\implies \phi^{\mathrm{UL}} = \phi^{\left(i +1\right)}  \\
    F\left(\phi^{\left(i+1\right)}\right)< -\epsilon &\implies \phi^{\mathrm{LL}} = \phi^{\left(i + 1\right)} \\
    - \epsilon < F\left(\phi^{i+1}\right)< \epsilon &\implies \phi_{\mathrm{HP}} \approx \phi^{\left(i + 1\right)}
\end{aligned}
$$
With our initial limits: Initial lower limit (LL), $\phi^{\left(0\right)}_{\mathrm{LL}} = \phi_{\mathrm{pzc}}$, and initial upper limit (UL) $\phi^{\left(0\right)}_{\mathrm{UL}}= \phi_{\mathrm{M}}$, respectively. $\epsilon$ denotes the error threshold. 

Finally, the maximum number of iterations, $N_{\mathrm{max}}$,
$$
\begin{aligned}
    N_{\mathrm{max}} = \left\lceil\log_2\left(\frac{\phi^{(0)}_{\mathrm{UL}}-\phi^{(0)}_{\mathrm{LL}}}{\epsilon}\right)\right\rceil
\end{aligned}
$$
where $\lceil \cdot \rceil$ denotes the ceiling function. Please note that $\log_2$ is the $2$ logarithm. For instance, the default values of this code, we use $\phi^{(0)}_{\mathrm{UL}}-\phi^{(0)}_{\mathrm{LL}} = 0.20$ V and an error threshold (converted from thermal voltage) of $\epsilon = 0.02569 \, \mathrm{V}\times10^{−12} \approx 2.57\times 10^{-14} V$. Plugging the values in result in $N_{\mathrm{max}} = \left\lceil\log_2\left(\frac{0.20\, \mathrm{V}}{2.57\times 10^{-14} \, \mathrm{V}}\right)\right\rceil \approx \left\lceil 42.82\right\rceil = 43$.


A quick note: Improving this scheme is a fun and, in principal, easy task. One can try to improve the starting lower limit or upper limit $\phi_{\mathrm{pzc}} \leq \phi_{\mathrm{HP}} \leq \phi_{\mathrm{M}}$. In the code below, you can see that we are using the value of Gouy–Chapman at, what would be called, the Helmholtz plane as our initial upper bound due to it satisfying $\phi_{\mathrm{HP}}^{(\mathrm{GCS})} \leq \phi^{(\mathrm{GC})}(x_{\mathrm{HP}}) \leq \phi_{\mathrm{M}}$, moving the initial upper bound closer to the value we are searching for (these inequalities are not shown here). The tighter the bounds, the faster the convergence. If the upper bound and lower bound meet before running the solution, then that limit would correspond to an exact solution of the Gouy–Chapman–Stern model.

In [29]:
def solve_U_HP_from_UM(
    UM: float, alpha: float, X_HP: float,
    tol: float = 1e-12, maxiter: int = 200,
    use_gc_upper: bool = True,
    u0: float | None = None
):
    if UM <= 0.0:
        return 0.0, 0

    
    L = 0.0 # Inital lower bracket
    U = UM  # Initial upper bracket

    if use_gc_upper:
        U_try = U_GC_scalar(X_HP, UM)       # We use the Gouy–Chapman solution as an initial guess for style points
        if 0.0 < U_try < UM:
            FU_try = UM - 2.0 * alpha * math.sinh(U_try / 2.0) - U_try
            if FU_try < 0.0:
                U = U_try

    u = 0.5 * (L + U) if (u0 is None) else float(u0)
    if not (L < u < U):
        u = 0.5 * (L + U)

    iters = 0
    for _ in range(maxiter):
        iters += 1
        if (U - L) < tol:
            break

        h = 0.5 * u
        sh = math.sinh(h)
        ch = math.cosh(h)

        Fu = UM - 2.0 * alpha * sh - u
        Fpu = -alpha * ch - 1.0

        u_new = u - Fu / Fpu

        if not (L < u_new < U):
            u_new = 0.5 * (L + U)

        Fu_new = UM - 2.0 * alpha * math.sinh(0.5 * u_new) - u_new

        if Fu_new > 0.0:
            L = u_new       # Positive Residual function means our guess is too low, so we update the lower bracket
        else:
            U = u_new       # Negative Residual function means our guess is too high, so we update the upper bracket   

        u = u_new

    return u, iters

# Interactive Gouy–Chapman–Stern sweep

The cells below mirror the interactive setup from the Gouy–Chapman notebook, but use the Stern boundary condition. The applied metal potential is split into a linear compact-layer drop and a diffuse Gouy–Chapman drop at the Helmholtz plane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C_PER_M2_TO_UC_PER_CM2 = 100.0
F_PER_M2_TO_UF_PER_CM2 = 100.0


def solve_U_HP_signed_from_VM(p: GCS_SI_Params, V_M: float):
    """Return dimensionless U_HP for a possibly signed applied potential V_M."""
    validate_params_si(p)
    lam_D = compute_lambda_D(p)
    X_HP = p.delta_H / lam_D
    alpha = (p.eps_S / p.eps_H) * X_HP

    sgn = 1.0 if V_M >= 0 else -1.0
    UM_abs = abs(V_to_U(p, V_M))

    U_HP_abs, iters = solve_U_HP_from_UM(
        UM_abs,
        alpha,
        X_HP,
        tol=p.tol,
        maxiter=p.maxiter,
        use_gc_upper=True,
    )
    return sgn * U_HP_abs, iters


def V_HP_from_VM(p: GCS_SI_Params, V_M):
    """Helmholtz-plane potential in volts for scalar or array V_M."""
    V_M_arr = np.asarray(V_M, dtype=float)
    out = np.empty_like(V_M_arr, dtype=float)

    it = np.nditer(V_M_arr, flags=["multi_index"])
    for v in it:
        U_HP, _ = solve_U_HP_signed_from_VM(p, float(v))
        out[it.multi_index] = U_to_V(p, U_HP)

    if np.ndim(V_M) == 0:
        return out.item()
    return out


def U_GC_vectorized(X, U0):
    X = np.asarray(X, dtype=float)
    y = np.exp(-X) * np.tanh(U0 / 4.0)
    # Avoid exactly +/-1 in extreme parameter choices.
    y = np.clip(y, -1.0 + 1e-15, 1.0 - 1e-15)
    return 4.0 * np.arctanh(y)


def psi_GCS(x, p: GCS_SI_Params, V_M: float):
    """GCS potential profile in volts from the metal surface through the diffuse layer."""
    validate_params_si(p)
    x = np.asarray(x, dtype=float)
    lam_D = compute_lambda_D(p)
    V_HP = V_HP_from_VM(p, V_M)
    U_HP = V_to_U(p, V_HP)

    psi = np.empty_like(x, dtype=float)
    stern = x <= p.delta_H
    diffuse = ~stern

    psi[stern] = V_M + (V_HP - V_M) * (x[stern] / p.delta_H)
    X = (x[diffuse] - p.delta_H) / lam_D
    psi[diffuse] = U_to_V(p, U_GC_vectorized(X, U_HP))
    return psi


def cation_concentration_GCS(x, p: GCS_SI_Params, psi):
    """Cation concentration. The compact Stern layer is ion-free and shown as NaN."""
    x = np.asarray(x, dtype=float)
    psi = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi)
    c = p.c_bulk * np.exp(-U)
    c[x <= p.delta_H] = np.nan
    return c


def anion_concentration_GCS(x, p: GCS_SI_Params, psi):
    """Anion concentration. The compact Stern layer is ion-free and shown as NaN."""
    x = np.asarray(x, dtype=float)
    psi = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi)
    c = p.c_bulk * np.exp(+U)
    c[x <= p.delta_H] = np.nan
    return c


def surface_free_charge_GCS(p: GCS_SI_Params, V_M):
    """Metal free charge density in C/m^2."""
    V_M_arr = np.asarray(V_M, dtype=float)
    V_HP = V_HP_from_VM(p, V_M_arr)
    sigma = C_H(p) * (V_M_arr - V_HP)
    if np.ndim(V_M) == 0:
        return sigma.item()
    return sigma


def C_GCS(p: GCS_SI_Params, V_M):
    """Differential double-layer capacitance in F/m^2 as a function of applied potential."""
    V_M_arr = np.asarray(V_M, dtype=float)
    V_HP = V_HP_from_VM(p, V_M_arr)
    Cg = C_GC(p, V_HP)
    Ct = C_total(C_H(p), Cg)
    if np.ndim(V_M) == 0:
        return float(Ct)
    return Ct


def make_gcs_params_and_VM_from_input(input_name, value, base_params, base_VM):
    delta_H = base_params.delta_H
    eps_H = base_params.eps_H
    eps_S = base_params.eps_S
    c_bulk = base_params.c_bulk
    T = base_params.T
    V_M = base_VM

    if input_name == "Concentration":
        c_bulk = value
    elif input_name == "Temperature":
        T = value
    elif input_name == "Bulk dielectric constant":
        eps_S = value
    elif input_name == "Stern dielectric constant":
        eps_H = value
    elif input_name == "Stern thickness":
        delta_H = value * 1e-9  # input is nm
    elif input_name == "Applied Potential":
        V_M = value
    else:
        raise ValueError(f"Unsupported input_name: {input_name}")

    p = GCS_SI_Params(
        delta_H=delta_H,
        eps_H=eps_H,
        eps_S=eps_S,
        c_bulk=c_bulk,
        T=T,
        tol=base_params.tol,
        maxiter=base_params.maxiter,
    )
    validate_params_si(p)
    return p, V_M

# Define plotting setup from parameter list

In [31]:
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

def plot_gcs_profiles_sigma_cap(
    input_name,
    input_values,
    V_M,
    x_max_nm,
    npts,
    base_concentration,
    base_temperature,
    base_eps_S,
    base_eps_H,
    base_delta_H_nm,
    phi_min,
    phi_max,
    sigma_npts=250,
):
    if phi_min >= phi_max:
        raise ValueError("phi_min must be smaller than phi_max")
    if x_max_nm <= 0:
        raise ValueError("x_max_nm must be > 0")
    if npts < 3:
        raise ValueError("npts must be at least 3")

    base_params = GCS_SI_Params(
        delta_H=base_delta_H_nm * 1e-9,
        eps_H=base_eps_H,
        eps_S=base_eps_S,
        c_bulk=base_concentration,
        T=base_temperature,
    )
    validate_params_si(base_params)

    if base_params.delta_H * 1e9 >= x_max_nm:
        raise ValueError("δ_HP must be smaller than x_max.")

    phi_grid = np.linspace(phi_min, phi_max, sigma_npts)

    x_full_nm = np.linspace(0.0, x_max_nm, npts)
    x_full_m = x_full_nm * 1e-9

    fig = plt.figure(figsize=(12, 7), constrained_layout=True)
    gs = fig.add_gridspec(2, 6)

    ax1 = fig.add_subplot(gs[0, 0:2])
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax3 = fig.add_subplot(gs[0, 4:6])
    ax4 = fig.add_subplot(gs[1, 0:3])
    ax5 = fig.add_subplot(gs[1, 3:6])

    ax1_zoom = ax1.inset_axes([0.56, 0.56, 0.40, 0.36])
    ax2_zoom = ax2.inset_axes([0.56, 0.56, 0.40, 0.36])
    ax3_zoom = ax3.inset_axes([0.56, 0.56, 0.40, 0.36])

    sweeping_applied_potential = input_name == "Applied Potential"

    ncurves = len(input_values)
    tvals = np.array([0.0]) if ncurves == 1 else np.linspace(0.0, 1.0, ncurves, endpoint=False)

    def blend_with_white(rgb, strength=0.65):
        rgb = np.array(rgb[:3], dtype=float)
        return tuple((1 - strength) * np.ones(3) + strength * rgb)

    def mix(c1, c2, t):
        c1 = np.array(c1, dtype=float)
        c2 = np.array(c2, dtype=float)
        return tuple((1 - t) * c1 + t * c2)

    start_gray = np.array([0.15, 0.15, 0.15])
    end_gray = np.array([0.65, 0.65, 0.65])

    potential_colors = (
        [tuple(start_gray)]
        if ncurves == 1
        else [mix(start_gray, end_gray, i / (ncurves - 1)) for i in range(ncurves)]
    )
    cation_colors = [blend_with_white((1.0, 0.0, 0.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    anion_colors = [blend_with_white((0.0, 0.0, 1.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    sigma_colors = [blend_with_white((0.0, 0.55, 0.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    cap_colors = [blend_with_white((0.5, 0.0, 0.5, 1.0), 0.25 + 0.65 * t) for t in tvals]

    # First build all curve-specific parameter sets.
    # This lets the largest Stern thickness determine the common zoom window.
    curve_data = []
    zoom_xmax_seen = 0.0

    for val in input_values:
        p_tmp, V_M_tmp = make_gcs_params_and_VM_from_input(
            input_name=input_name,
            value=val,
            base_params=base_params,
            base_VM=V_M,
        )

        delta_H_nm_tmp = p_tmp.delta_H * 1e9

        if delta_H_nm_tmp >= x_max_nm:
            raise ValueError("δ_HP must be smaller than x_max for every curve.")

        x_zoom_max_nm_tmp = 2.0 * delta_H_nm_tmp

        curve_data.append(
            (val, p_tmp, V_M_tmp, delta_H_nm_tmp, x_zoom_max_nm_tmp)
        )

        zoom_xmax_seen = max(zoom_xmax_seen, x_zoom_max_nm_tmp)

    # Common zoom window for the whole sweep.
    # The largest 2δ_HP determines the zoom length.
    x_zoom_max_nm_common = min(zoom_xmax_seen, x_max_nm)
    x_zoom_nm_common = np.linspace(0.0, x_zoom_max_nm_common, npts)
    x_zoom_m_common = x_zoom_nm_common * 1e-9

    for i, (val, p, V_M_this, delta_H_nm_this, x_zoom_max_nm_this) in enumerate(curve_data):

        # Use the same zoom grid for every sweep curve.
        x_zoom_nm = x_zoom_nm_common
        x_zoom_m = x_zoom_m_common

        psi_full = psi_GCS(x_full_m, p, V_M_this)
        ncat_full = cation_concentration_GCS(x_full_m, p, psi_full)
        nani_full = anion_concentration_GCS(x_full_m, p, psi_full)

        psi_zoom = psi_GCS(x_zoom_m, p, V_M_this)
        ncat_zoom = cation_concentration_GCS(x_zoom_m, p, psi_zoom)
        nani_zoom = anion_concentration_GCS(x_zoom_m, p, psi_zoom)

        if input_name == "Concentration":
            label = f"{val:g} mM"
        elif input_name == "Temperature":
            label = f"{val:g} K"
        elif input_name == "Applied Potential":
            label = f"{val:g} V"
        elif input_name == "Bulk dielectric constant":
            label = rf"$\varepsilon_S = {val:g}$"
        elif input_name == "Stern dielectric constant":
            label = rf"$\varepsilon_{{HP}} = {val:g}$"
        elif input_name == "Stern thickness":
            label = rf"$\delta_{{HP}} = {val:g}$ nm"
        else:
            label = f"{input_name} = {val:g}"

        # Row 1: zoomed profiles
        ax1.plot(x_full_nm, psi_full, label=label, color=potential_colors[i], lw=2.5)
        ax2.plot(x_full_nm, ncat_full, label=label, color=cation_colors[i], lw=2)
        ax3.plot(x_full_nm, nani_full, label=label, color=anion_colors[i], lw=2)

        ax1_zoom.plot(x_zoom_nm, psi_zoom, color=potential_colors[i], lw=1.8)
        ax2_zoom.plot(x_zoom_nm, ncat_zoom, color=cation_colors[i], lw=1.6)
        ax3_zoom.plot(x_zoom_nm, nani_zoom, color=anion_colors[i], lw=1.6)

        # Row 3: unchanged sigma/capacitance panels
        sigma_vals = C_PER_M2_TO_UC_PER_CM2 * surface_free_charge_GCS(p, phi_grid)
        cap_vals = F_PER_M2_TO_UF_PER_CM2 * C_GCS(p, phi_grid)

        ax4.plot(phi_grid, sigma_vals, label=label, color=sigma_colors[i], lw=2)
        ax5.plot(phi_grid, cap_vals, label=label, color=cap_colors[i], lw=2)

        hp_nm = delta_H_nm_this
        zoom_end_nm = x_zoom_max_nm_common

        # Mark Helmholtz plane in both zoomed and full profile panels.
        for a in (ax1_zoom, ax2_zoom, ax3_zoom, ax1, ax2, ax3):
            a.axvline(hp_nm, color="0.75", lw=1, ls=":")

        # Mark the zoom window on the full-profile panels.
        for a in (ax1, ax2, ax3):
            a.axvspan(0.0, zoom_end_nm, color="0.92", alpha=0.45, zorder=0)


    # Row 2 labels
    ax1.set_title("Electric potential")
    ax1.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax1.set_ylabel("Electric potential (V)", fontsize=12)

    ax2.set_title("Cation density")
    ax2.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax2.set_ylabel("Cation density (mol/m$^3$)", fontsize=12)

    ax3.set_title("Anion density")
    ax3.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax3.set_ylabel("Anion density (mol/m$^3$)", fontsize=12)

    # Row 3 labels
    ax4.set_title("Surface free charge density")
    ax4.set_xlabel(r"$\phi_M - \phi_{pzc}$ (V)", fontsize=14)
    ax4.set_ylabel(r"Surface free charge density ($\mu$C/cm$^2$)")

    ax5.set_title("GCS differential capacitance")
    ax5.set_xlabel(r"$\phi_M - \phi_{pzc}$ (V)", fontsize=14)
    ax5.set_ylabel(r"Double-layer capacitance ($\mu$F/cm$^2$)")

    for a in (ax1_zoom, ax2_zoom, ax3_zoom, ax1, ax2, ax3, ax4, ax5):
        a.grid(True)

    ax1.legend(fontsize=8, loc="lower left")
    ax2.legend(fontsize=8, loc="lower left")
    ax3.legend(fontsize=8, loc="lower left")

    if not sweeping_applied_potential:
        ax4.legend(fontsize=9)
        ax5.legend(fontsize=9)
    
    for a in (ax1_zoom, ax2_zoom, ax3_zoom):
        a.grid(True, alpha=0.25)
        a.tick_params(axis="both", labelsize=7)
        a.set_xlabel("")
        a.set_ylabel("")
        a.set_title("")
        a.set_xlim(0.0, x_zoom_max_nm_common)

    # Make insets true x-zooms, not independent y-rescaled plots.
    ax1_zoom.set_ylim(ax1.get_ylim())
    ax2_zoom.set_ylim(ax2.get_ylim())
    ax3_zoom.set_ylim(ax3.get_ylim())

    # Label Stern and diffuse regions in the inset windows.
    for a in (ax1_zoom, ax2_zoom, ax3_zoom):

        ymin, ymax = a.get_ylim()
        y_text = ymax - 0.08 * (ymax - ymin)

        a.text(
            0.25 * zoom_xmax_seen,
            y_text,
            "Stern\nlayer",
            ha="center",
            va="top",
            fontsize=7,
            fontweight="bold",
            color="0.20",
        )

        a.text(
            0.75 * zoom_xmax_seen,
            y_text,
            "Diffuse\nlayer",
            ha="center",
            va="top",
            fontsize=7,
            fontweight="bold",
            color="0.20",
        )

    mark_inset(ax1, ax1_zoom, loc1=1, loc2=3, fc="none", ec="0.45", lw=0.9)
    mark_inset(ax1, ax1_zoom, loc1=2, loc2=4, fc="none", ec="0.45", lw=0.9)

    mark_inset(ax2, ax2_zoom, loc1=1, loc2=3, fc="none", ec="0.45", lw=0.9)
    mark_inset(ax2, ax2_zoom, loc1=2, loc2=4, fc="none", ec="0.45", lw=0.9)

    mark_inset(ax3, ax3_zoom, loc1=1, loc2=3, fc="none", ec="0.45", lw=0.9)
    mark_inset(ax3, ax3_zoom, loc1=2, loc2=4, fc="none", ec="0.45", lw=0.9)

    return fig

# Define error box

In [32]:
import ipywidgets as widgets


def show_error(message):
    display(widgets.HTML(
        f"""
        <div style="
            color:#8a1f11;
            background:#fff3f0;
            border:1px solid #f0b4a8;
            border-radius:6px;
            padding:10px;
            font-size:14px;
            max-width:700px;
        ">
            <b>Error:</b> {message}
        </div>
        """
    ))

# Make User Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

LABEL_WIDTH = "190px"
BOX_WIDTH = "150px"


def make_row(label_html, widget, fontsize="16px"):
    widget.layout = widgets.Layout(width=BOX_WIDTH)
    label = widgets.HTML(
        value=f"""
        <span style="font-size:{fontsize}; line-height:1.2;">
            {label_html}
        </span>
        """,
        layout=widgets.Layout(width=LABEL_WIDTH),
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(align_items="center"))


input_dropdown = widgets.Dropdown(
    options=[
        "Concentration",
        "Temperature",
        "Bulk dielectric constant",
        "Stern dielectric constant",
        "Stern thickness",
        "Applied Potential",
    ],
    value="Stern thickness",
    layout=widgets.Layout(width=BOX_WIDTH),
)
input_row = make_row("Input:", input_dropdown)

values_text = widgets.Text(value="0.03, 0.1, 0.3", placeholder="comma-separated values")
values_row = make_row("Sweep:", values_text)

VM_box = widgets.FloatText(value=0.1)
VM_row = make_row("ϕ<sub>M</sub> − ϕ<sub>pzc</sub> (V):", VM_box)

phi_min_box = widgets.FloatText(value=-0.3)
phi_min_row = make_row("ϕ<sub>min</sub> (V):", phi_min_box)

phi_max_box = widgets.FloatText(value=0.3)
phi_max_row = make_row("ϕ<sub>max</sub> (V):", phi_max_box)

xmax_box = widgets.FloatText(value=5.0)
xmax_row = make_row("x<sub>max</sub> (nm):", xmax_box)

base_c_box = widgets.FloatText(value=10.0)
base_c_row = make_row("base c (mM):", base_c_box)

base_T_box = widgets.FloatText(value=298.15)
base_T_row = make_row("base T (K):", base_T_box)

base_eps_S_box = widgets.FloatText(value=78.5)
base_eps_S_row = make_row("base ε<sub>S</sub>:", base_eps_S_box)

base_eps_H_box = widgets.FloatText(value=6.0)
base_eps_H_row = make_row("base ε<sub>HP</sub>:", base_eps_H_box)

base_delta_H_box = widgets.FloatText(value=0.03)
base_delta_H_row = make_row("base δ<sub>HP</sub> (nm):", base_delta_H_box)

plot_button = widgets.Button(
    description="Plot",
    button_style="success",
    layout=widgets.Layout(width="120px"),
)
plot_row = widgets.HBox([widgets.HTML(layout=widgets.Layout(width=LABEL_WIDTH)), plot_button])

out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))
npts = 300

base_header = widgets.HTML(
    """
    <div style="
        font-size:14px;
        font-weight:600;
        margin-top:8px;
        margin-bottom:4px;
    ">
        Base values used for parameters not being swept:
    </div>
    """
)


def update_visible_base_inputs(*args):
    swept = input_dropdown.value
    base_c_row.layout.display = "" if swept != "Concentration" else "none"
    base_T_row.layout.display = "" if swept != "Temperature" else "none"
    base_eps_S_row.layout.display = "" if swept != "Bulk dielectric constant" else "none"
    base_eps_H_row.layout.display = "" if swept != "Stern dielectric constant" else "none"
    base_delta_H_row.layout.display = "" if swept != "Stern thickness" else "none"
    VM_row.layout.display = "none" if swept == "Applied Potential" else ""


def on_plot_clicked(b):
    out.clear_output(wait=True)
    plt.close("all")

    try:
        input_values = np.array([float(v.strip()) for v in values_text.value.split(",") if v.strip()])
        if input_values.size == 0:
            raise ValueError("Please enter at least one sweep value.")
        if xmax_box.value <= 0:
            raise ValueError("x max must be > 0 nm.")
        if npts < 3:
            raise ValueError("Points must be at least 3.")
        if phi_min_box.value >= phi_max_box.value:
            raise ValueError("phi min must be smaller than phi max.")

        fig = plot_gcs_profiles_sigma_cap(
            input_name=input_dropdown.value,
            input_values=input_values,
            V_M=VM_box.value,
            x_max_nm=xmax_box.value,
            npts=npts,
            base_concentration=base_c_box.value,
            base_temperature=base_T_box.value,
            base_eps_S=base_eps_S_box.value,
            base_eps_H=base_eps_H_box.value,
            base_delta_H_nm=base_delta_H_box.value,
            phi_min=phi_min_box.value,
            phi_max=phi_max_box.value,
            sigma_npts=npts,
        )

        with out:
            display(fig)
        plt.close(fig)

    except ValueError as err:
        plt.close("all")
        out.clear_output(wait=True)
        with out:
            show_error(f"Invalid input: {err}")

    except Exception as err:
        plt.close("all")
        out.clear_output(wait=True)
        with out:
            show_error(f"Unexpected error: {err}")


try:
    input_dropdown.unobserve(update_visible_base_inputs, names="value")
except Exception:
    pass
input_dropdown.observe(update_visible_base_inputs, names="value")
plot_button.on_click(on_plot_clicked, remove=True)
plot_button.on_click(on_plot_clicked)
update_visible_base_inputs()

controls = widgets.VBox(
    [
        input_row,
        values_row,
        VM_row,
        phi_min_row,
        phi_max_row,
        xmax_row,
        base_header,
        base_c_row,
        base_T_row,
        base_eps_S_row,
        base_eps_H_row,
        base_delta_H_row,
        plot_row,
    ],
    layout=widgets.Layout(width="370px", min_width="370px"),
)

output_panel = widgets.Box([out], layout=widgets.Layout(flex="1 1 0%", width="auto"))
ui = widgets.HBox([controls, output_panel], layout=widgets.Layout(width="100%", align_items="flex-start"))
display(ui)

# Plot Experimental and Gouy–Chapman–Stern together


In [34]:
from pathlib import Path
import numpy as np

VALETTE_FILES = {
    5.0: "Valette1982_5mM.txt",
    10.0: "Valette1982_10mM.txt",
    20.0: "Valette1982_20mM.txt",
    40.0: "Valette1982_40mM.txt",
    100.0: "Valette1982_100mM.txt",
}


def load_valette_txt(filepath):
    E_vals = []
    C_vals = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue
            if line.startswith("%") or line.startswith("#"):
                continue

            parts = line.replace(",", " ").split()
            if len(parts) < 2:
                continue

            try:
                E_vals.append(float(parts[0]))
                C_vals.append(float(parts[1]))
            except ValueError:
                # Allow a small amount of header text.
                continue

    if not E_vals:
        raise ValueError(f"No numeric data found in {filepath}")

    return np.array(E_vals, dtype=float), np.array(C_vals, dtype=float)


def find_valette_data_dir():
    """
    Try the folder layout used by the original Gouy–Chapman notebook, while
    also allowing the data folder to sit next to this notebook.
    """
    candidates = [
        Path.cwd() / "Valette1982",
        Path.cwd().parent / "Valette1982",
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/Valette1982"),
        Path("/mnt/data"),
    ]

    for directory in candidates:
        if all((directory / filename).exists() for filename in VALETTE_FILES.values()):
            return directory

    for directory in candidates:
        if any((directory / filename).exists() for filename in VALETTE_FILES.values()):
            return directory

    return candidates[0]


DATA_DIR = find_valette_data_dir()


def load_all_valette_data(data_dir=DATA_DIR, quiet=False):
    data = {}
    missing = []

    for conc_mM, filename in VALETTE_FILES.items():
        filepath = Path(data_dir) / filename

        if filepath.exists():
            E, C = load_valette_txt(filepath)
            data[conc_mM] = {"E": E, "C": C, "path": filepath}
        else:
            missing.append(filepath)

    if missing and not quiet:
        print("Missing Valette files:")
        for filepath in missing:
            print(f"  {filepath}")

    return data, missing


VALETTE_DATA, VALETTE_MISSING = load_all_valette_data()


In [35]:
import matplotlib.pyplot as plt
import numpy as np

def make_gcs_params_for_valette(
    concentration_mM,
    temperature,
    eps_S,
    eps_H,
    delta_H_nm,
):
    p = GCS_SI_Params(
        delta_H=delta_H_nm * 1e-9,
        eps_H=eps_H,
        eps_S=eps_S,
        c_bulk=concentration_mM,
        T=temperature,
    )
    validate_params_si(p)
    return p


def plot_gcs_valette_comparison(
    concentrations_mM,
    temperature=298.15,
    eps_S=78.5,
    eps_H=6.0,
    delta_H_nm=0.03,
    phi_pzc_sce=-0.92,
    E_min=-1.02,
    E_max=-0.82,
    npts=350,
    show_data=True,
    show_missing_note=True,
):

    if E_min >= E_max:
        raise ValueError("E min must be smaller than E max.")
    if delta_H_nm <= 0:
        raise ValueError("Stern thickness must be > 0 nm.")
    if eps_H <= 0 or eps_S <= 0:
        raise ValueError("Dielectric constants must be > 0.")
    if eps_S <= eps_H:
        raise ValueError("Bulk/solution dielectric ε_S should be larger than Stern-layer dielectric ε_H.")
    if temperature <= 0:
        raise ValueError("Temperature must be > 0 K.")
    if npts < 3:
        raise ValueError("npts must be at least 3.")


    concentrations_mM = [float(c) for c in concentrations_mM]
    if len(concentrations_mM) == 0:
        raise ValueError("Please enter at least one concentration.")

    E_grid = np.linspace(E_min, E_max, npts)
    V_model = E_grid - phi_pzc_sce

    fig = plt.figure(figsize=(12, 8.5), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, height_ratios=[1.35, 1.0])

    ax = fig.add_subplot(gs[0, :])     # top: total GCS + Valette
    ax_H = fig.add_subplot(gs[1, 0])   # bottom left: Helmholtz
    ax_GC = fig.add_subplot(gs[1, 1])  # bottom right: Gouy--Chapman

    cmap = plt.cm.magma_r
    ncurves = len(concentrations_mM)
    colors = [cmap(0.65)] if ncurves == 1 else [cmap(t) for t in np.linspace(0.15, 0.90, ncurves)]

    for color, conc in zip(colors, concentrations_mM):
        p = make_gcs_params_for_valette(
            concentration_mM=conc,
            temperature=temperature,
            eps_S=eps_S,
            eps_H=eps_H,
            delta_H_nm=delta_H_nm,
        )

        cap_model = F_PER_M2_TO_UF_PER_CM2 * C_GCS(p, V_model)

        C_H_val = F_PER_M2_TO_UF_PER_CM2 * C_H(p)

        V_HP_model = V_HP_from_VM(p, V_model)
        C_GC_model = F_PER_M2_TO_UF_PER_CM2 * C_GC(p, V_HP_model)

        ax.plot(
            E_grid,
            cap_model,
            lw=2.6,
            color=color,
            solid_capstyle="round",
            label=rf"GCS {conc:g} mM",
        )

        ax_H.plot(
            E_grid,
            np.full_like(E_grid, C_H_val),
            lw=2.4,
            color=color,
            solid_capstyle="round",
            label=rf"{conc:g} mM",
        )

        ax_GC.plot(
            E_grid,
            C_GC_model,
            lw=2.4,
            color=color,
            solid_capstyle="round",
            label=rf"{conc:g} mM",
        )

        if show_data and conc in VALETTE_DATA:
            data = VALETTE_DATA[conc]
            darker = tuple(np.clip(np.array(color[:3]) * 0.72, 0, 1))

            scatter_kwargs = dict(
                s=36,
                marker="o",
                facecolor=darker,
                edgecolor="white",
                linewidth=0.7,
                alpha=0.95,
                zorder=3,
            )

            ax.scatter(
                data["E"],
                data["C"],
                label=rf"Valette 1982 {conc:g} mM",
                **scatter_kwargs,
            )

            ax_H.scatter(
                data["E"],
                data["C"],
                label="_nolegend_",
                **scatter_kwargs,
            )

            ax_GC.scatter(
                data["E"],
                data["C"],
                label="_nolegend_",
                **scatter_kwargs,
            )

    # Gouy–Chapman–Stern
    ax.axvline(phi_pzc_sce, color="0.35", lw=1.2, ls="--")
    ax.set_title("Gouy–Chapman–Stern capacitance compared with Valette 1982", fontsize=18, pad=12)
    ax.set_xlabel(r"ϕ$_{\mathrm{M}}$ (V)", fontsize=14)
    ax.set_ylabel(r"Differential capacitance ($\mu$F/cm$^2$)", fontsize=14)

    # Helmholtz
    ax_H.axvline(phi_pzc_sce, color="0.35", lw=1.2, ls="--")
    ax_H.set_title("Helmholtz capacitance", fontsize=15, pad=10)
    ax_H.set_xlabel(r"ϕ$_{\mathrm{M}}$ (V)", fontsize=13)
    ax_H.set_ylabel(r"$C_{\mathrm{H}}$ ($\mu$F/cm$^2$)", fontsize=13)

    # Gouy–Chapman
    ax_GC.axvline(phi_pzc_sce, color="0.35", lw=1.2, ls="--")
    ax_GC.set_title("Diffuse Gouy–Chapman capacitance", fontsize=15, pad=10)
    ax_GC.set_xlabel(r"ϕ$_{\mathrm{M}}$ (V)", fontsize=13)
    ax_GC.set_ylabel(r"$C_{\mathrm{GC}}(\phi_{\mathrm{HP}})$ ($\mu$F/cm$^2$)", fontsize=13)

    for a in (ax, ax_H, ax_GC):
        a.grid(True, alpha=0.25)
        a.set_axisbelow(True)
        a.spines["top"].set_visible(False)
        a.spines["right"].set_visible(False)
        a.tick_params(axis="both", labelsize=11)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(fontsize=8.5, frameon=True, fancybox=True, framealpha=0.92, edgecolor="0.85", ncols=2)

    handles_H, labels_H = ax_H.get_legend_handles_labels()
    if handles_H:
        ax_H.legend(fontsize=8, frameon=True, fancybox=True, framealpha=0.92, edgecolor="0.85")

    handles_GC, labels_GC = ax_GC.get_legend_handles_labels()
    if handles_GC:
        ax_GC.legend(fontsize=8, frameon=True, fancybox=True, framealpha=0.92, edgecolor="0.85")

    if show_missing_note and show_data and VALETTE_MISSING:
        available = ", ".join(f"{c:g} mM" for c in sorted(VALETTE_DATA))
        if available:
            note = f"Loaded Valette data: {available}."
        else:
            note = "No Valette data files were found. Put the Valette1982 folder next to this notebook or one folder above it."
        fig.text(0.01, 0.01, note, ha="left", va="bottom", fontsize=9, color="0.35")

    return fig


# Interactive Valette comparison UI

Use this panel to tune the Stern-layer parameters and compare the resulting GCS capacitance curves to the Valette data.

In [36]:
import ipywidgets as widgets
from IPython.display import display

VAL_LABEL_WIDTH = "190px"
VAL_BOX_WIDTH = "150px"


def make_valette_row(label_html, widget, fontsize="16px"):
    widget.layout = widgets.Layout(width=VAL_BOX_WIDTH)
    label = widgets.HTML(
        value=f"""
        <span style="font-size:{fontsize}; line-height:1.2;">
            {label_html}
        </span>
        """,
        layout=widgets.Layout(width=VAL_LABEL_WIDTH),
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(align_items="center"))

valette_T_box = widgets.FloatText(value=298.15)
valette_T_row = make_valette_row("T (K):", valette_T_box)

valette_eps_S_box = widgets.FloatText(value=78.5)
valette_eps_S_row = make_valette_row("ε<sub>S</sub>:", valette_eps_S_box)

valette_eps_H_box = widgets.FloatText(value=6.0)
valette_eps_H_row = make_valette_row("ε<sub>HP</sub>:", valette_eps_H_box)

valette_delta_H_box = widgets.FloatText(value=0.03)
valette_delta_H_row = make_valette_row("δ<sub>HP</sub> (nm):", valette_delta_H_box)

valette_phi_pzc_box = widgets.FloatText(value=-0.92)
valette_phi_pzc_row = make_valette_row("ϕ<sub>pzc</sub> (V):", valette_phi_pzc_box)

valette_E_min_box = widgets.FloatText(value=-1.12)
valette_E_min_row = make_valette_row("ϕ<sub>min</sub> (V):", valette_E_min_box)

valette_E_max_box = widgets.FloatText(value=-0.72)
valette_E_max_row = make_valette_row("ϕ<sub>max</sub> (V):", valette_E_max_box)

valette_plot_button = widgets.Button(
    description="Plot comparison",
    button_style="success",
    layout=widgets.Layout(width="150px"),
)
valette_plot_row = widgets.HBox([widgets.HTML(layout=widgets.Layout(width=VAL_LABEL_WIDTH)), valette_plot_button])

valette_out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))


def on_valette_gcs_plot_clicked(b):
    valette_out.clear_output(wait=True)
    plt.close("all")

    try:
        concentrations = [5.0, 10.0, 20.0, 40.0, 100.0]
        fig = plot_gcs_valette_comparison(
            concentrations_mM=concentrations,
            temperature=valette_T_box.value,
            eps_S=valette_eps_S_box.value,
            eps_H=valette_eps_H_box.value,
            delta_H_nm=valette_delta_H_box.value,
            phi_pzc_sce=valette_phi_pzc_box.value,
            E_min=valette_E_min_box.value,
            E_max=valette_E_max_box.value,
            npts=350,
        )

        with valette_out:
            display(fig)
        plt.close(fig)

    except ValueError as err:
        plt.close("all")
        valette_out.clear_output(wait=True)
        with valette_out:
            show_error(f"Invalid input: {err}")

    except Exception as err:
        plt.close("all")
        valette_out.clear_output(wait=True)
        with valette_out:
            show_error(f"Unexpected error: {err}")


try:
    valette_plot_button.on_click(on_valette_gcs_plot_clicked, remove=True)
except Exception:
    pass
valette_plot_button.on_click(on_valette_gcs_plot_clicked)

valette_controls = widgets.VBox(
    [
        valette_T_row,
        valette_eps_S_row,
        valette_eps_H_row,
        valette_delta_H_row,
        valette_phi_pzc_row,
        valette_E_min_row,
        valette_E_max_row,
        valette_plot_row,
    ],
    layout=widgets.Layout(width="390px", min_width="390px"),
)

valette_output_panel = widgets.Box([valette_out], layout=widgets.Layout(flex="1 1 0%", width="auto"))
valette_ui = widgets.HBox([valette_controls, valette_output_panel], layout=widgets.Layout(width="100%", align_items="flex-start"))
display(valette_ui)
